<a href="https://colab.research.google.com/github/jameshphan-png/Group-Exercise-Agentic-AI-in-Customer-Service-Sales-/blob/dev/Customer_Service_Group_Exercise_Part_1_Order_Status.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTANT INFORMATION, READ BELOW**

In [ ]:
#THE FOLLOWING CODE IS AN ORDER STATUS CHATBOT, WHICH IS AN AGENTIC CUSTOMER SERVICE BOT THAT RESPONDS TO INQUIRIES DEDICATED TO THIS SEGMENT.

# ============================================================
#  Acme Corp — Agentic Customer Service Chatbot
#  Uses GeminiAPI - Based on 2.5 Flash Lite
# ============================================================

**Set-Up Company Details & Role Prompt (Order Status)**

*   API Call
*   Chat Conversation using LangGraph as memory > Logging conversation in a json file
*   Architecture focuses on "order_status" from the INTENT_KEYWORDS function, which retrieves the status of the order, promptly asking the customer for their Order ID before revealing transparency

**Due the nature of the project's API requests being rate-limited. It is recommended that you refer to the other project file to "run the code"**

Alternative Project File: https://colab.research.google.com/drive/1X4yUyMXIDnqAEQqudRU92dmMT-l9IcI9?usp=sharing

In [1]:
# ── Install & Imports ─────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "google-genai", "langgraph", "langchain-core", "-q"], check=True)

import os, json, re, time
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

from google import genai
from google.colab import userdata

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# ── Set-up Gemini API Client ──────────────────────────────────────────────────
api_key = userdata.get('part1')
client  = genai.Client(api_key=api_key)
MODEL   = "gemini-2.5-flash-lite"

# ── Company Details ───────────────────────────────────────────────────────────
COMPANY_CONFIG = {
    "name"          : "Acme Corp",
    "industry"      : "E-Commerce",
    "support_email" : "support@acmecorp.com",
    "support_hours" : "Monday-Friday, 9 AM - 6 PM EST",
    "return_policy" : "30-day hassle-free returns",
    "website"       : "https://www.acmecorp.com",
}

# ── Order Database ────────────────────────────────────────────────────────────
ORDERS_DB = {
    "ORD-1001": {"status": "Shipped",    "item": "AlphaBook Pro Laptop",  "estimated_date": "March 21, 2026", "carrier": "FedEx", "tracking": "FX9284710234"},
    "ORD-1002": {"status": "Processing", "item": "GammaAir X Laptop",     "estimated_date": "March 25, 2026", "carrier": "UPS",   "tracking": None},
    "ORD-1003": {"status": "Delivered",  "item": "NanoEdge Flex Laptop",  "estimated_date": "March 15, 2026", "carrier": "USPS",  "tracking": "9400111899223456789012"},
    "ORD-1004": {"status": "Cancelled",  "item": "SpectraBook S Laptop",  "estimated_date": "N/A",            "carrier": "N/A",   "tracking": None},
}

# ── Product Catalogue ─────────────────────────────────────────────────────────
PRODUCTS = [
    {"name": "AlphaBook Pro",  "price": 1499, "best_for": "professionals, lightweight travel",   "highlights": "12th Gen Intel i7, 16 GB RAM, 1 TB SSD, 2-day shipping"},
    {"name": "GammaAir X",     "price": 1399, "best_for": "everyday performance, students",      "highlights": "AMD Ryzen 7, 32 GB DDR4, 512 GB NVMe SSD, thin & light"},
    {"name": "SpectraBook S",  "price": 2499, "best_for": "power users, video editing, 3D work", "highlights": "Intel Core i9, 64 GB RAM, 2 TB SSD"},
    {"name": "OmegaPro G17",   "price": 2199, "best_for": "gamers, high-refresh-rate displays",  "highlights": "Ryzen 9 5900HX, 32 GB RAM, 1 TB SSD, 17-inch 165 Hz display"},
    {"name": "NanoEdge Flex",  "price": 1699, "best_for": "creatives, 2-in-1 tablet use",        "highlights": "360-degree hinge, stylus support, OLED touch display, 2-day shipping"},
]

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = f"""
You are Alex, a friendly and knowledgeable customer support agent for {COMPANY_CONFIG['name']},
an {COMPANY_CONFIG['industry']} company that sells premium laptops.

YOUR RESPONSIBILITIES:
1. Help customers check the status of their orders.
2. Answer questions about products, specs, pricing, and availability.
3. Handle return, refund, and exchange requests.
4. Assist with billing enquiries.
5. Escalate account-related issues that require human access.

BEHAVIOUR RULES:
- Always be warm, professional, and concise.
- Address the customer by name once you learn it.
- Use the exact order details or product data provided in SYSTEM NOTEs — never invent information.
- For billing and account issues, let the customer know a specialist will follow up within 1 business day.
- For returns, remind the customer of the return policy before escalating.

COMPANY DETAILS:
- Support email  : {COMPANY_CONFIG['support_email']}
- Support hours  : {COMPANY_CONFIG['support_hours']}
- Return policy  : {COMPANY_CONFIG['return_policy']}
- Website        : {COMPANY_CONFIG['website']}
""".strip()

# ── Intent Detection ──────────────────────────────────────────────────────────
INTENT_KEYWORDS = {
    "order_status" : ["order", "status", "where is", "shipped", "shipping", "tracking", "delivered", "arrived", "ord-"],
    "product_info" : ["product", "laptop", "recommend", "feature", "spec", "price", "buy", "purchase", "which one", "compare"],
    "returns"      : ["return", "refund", "exchange", "damaged", "defective", "wrong item", "broken"],
    "billing"      : ["charge", "bill", "invoice", "payment", "paid", "receipt"],
    "account"      : ["login", "password", "account", "sign in", "verification"],
}

def detect_intent(message: str) -> str:
    msg = message.lower()
    for intent, keywords in INTENT_KEYWORDS.items():
        if any(kw in msg for kw in keywords):
            return intent
    return "general"

# ── Order Lookup ──────────────────────────────────────────────────────────────
def lookup_order(message: str) -> str | None:
    match = re.search(r'ORD-\d+', message, re.IGNORECASE)
    if not match:
        return None
    order_id = match.group().upper()
    if order_id not in ORDERS_DB:
        return f"[SYSTEM NOTE: Order {order_id} was not found in the database.]"
    o        = ORDERS_DB[order_id]
    tracking = f"Tracking: {o['tracking']}" if o['tracking'] else "Tracking number not yet assigned."
    return (
        f"[SYSTEM NOTE - Order details for {order_id}:\n"
        f"  Item: {o['item']} | Status: {o['status']} | "
        f"Est. Delivery: {o['estimated_date']} | Carrier: {o['carrier']} | {tracking}\n"
        f"Use these exact details in your reply.]"
    )

# ── Product Context Builder ───────────────────────────────────────────────────
def build_product_context() -> str:
    lines = ["[SYSTEM NOTE - Available products:"]
    for p in PRODUCTS:
        lines.append(f"  - {p['name']} / ${p['price']} / Best for: {p['best_for']} / {p['highlights']}")
    lines.append("]")
    return "\n".join(lines)

# ── Conversation Logger ───────────────────────────────────────────────────────
class ConversationLogger:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.data = {
            "session_id" : session_id,
            "started_at" : datetime.now().isoformat(),
            "company"    : COMPANY_CONFIG["name"],
            "turns"      : 0,
            "messages"   : [],
        }

    def log(self, role: str, content: str, intent: str = ""):
        entry = {"timestamp": datetime.now().isoformat(), "role": role, "content": content}
        if intent:
            entry["intent"] = intent
        self.data["messages"].append(entry)
        if role == "user":
            self.data["turns"] += 1

    def save(self):
        self.data["ended_at"] = datetime.now().isoformat()
        log_file = "support_log.json"
        try:
            existing = json.load(open(log_file)) if os.path.exists(log_file) else []
            existing.append(self.data)
            json.dump(existing, open(log_file, "w"), indent=2)
            print(f"\n  Session saved → {log_file}  (ID: {self.session_id}, turns: {self.data['turns']})")
        except Exception as e:
            print(f"\n  Could not save log: {e}")

# ── LangGraph Setup ──────────────────────────────────────────────────────────

class AgentState(TypedDict):
    """
    Graph state persisted by MemorySaver after every turn.

    messages — full conversation history; the add_messages reducer
               *appends* new messages rather than overwriting the list,
               so we never lose prior turns.
    """
    messages: Annotated[list, add_messages]


def gemini_node(state: AgentState) -> dict:
    """
    Single graph node: converts LangGraph message objects into the flat
    prompt string Gemini expects, calls the API, and returns the new
    AIMessage. MemorySaver snapshots the updated state automatically.
    """
    full_prompt = SYSTEM_PROMPT + "\n\n"
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            full_prompt += f"Customer: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            full_prompt += f"Alex: {msg.content}\n"
    full_prompt += "Alex:"

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=full_prompt,
            )
            reply = response.text.strip()
            # Return only the *new* message — add_messages merges it into state
            return {"messages": [AIMessage(content=reply)]}
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def build_graph() -> StateGraph:
    """
    Compile a minimal START → alex → END graph with MemorySaver attached.

    MemorySaver checkpoints AgentState (the full message list) after every
    .invoke() call, keyed by thread_id.  Passing the same thread_id on the
    next turn restores the full conversation automatically — no manual
    history list needed.

    To persist across process restarts later, swap MemorySaver for
    SqliteSaver or PostgresSaver with zero other code changes.
    """
    memory  = MemorySaver()
    builder = StateGraph(AgentState)
    builder.add_node("alex", gemini_node)
    builder.add_edge(START, "alex")
    builder.add_edge("alex", END)
    return builder.compile(checkpointer=memory)


# One shared graph instance — MemorySaver lives inside it
GRAPH = build_graph()


def invoke_graph(thread_id: str, human_content: str) -> str:
    """
    Send one user turn to the graph and return Alex's reply.

    thread_id     — MemorySaver key; reusing the same ID restores history
    human_content — the (possibly context-augmented) user message
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = GRAPH.invoke(
        {"messages": [HumanMessage(content=human_content)]},
        config=config,
    )
    return result["messages"][-1].content


# ── Main Chat Session ─────────────────────────────────────────────────────────
def run_chat_session():
    # thread_id is the MemorySaver checkpoint key — unique per session
    thread_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    logger    = ConversationLogger(session_id=thread_id)

    print("=" * 60)
    print(f"  {COMPANY_CONFIG['name']}  -  Customer Support & Sales")
    print(f"  {COMPANY_CONFIG['support_hours']}")
    print(f"  {COMPANY_CONFIG['support_email']}")
    print(f"  Session ID (MemorySaver thread): {thread_id}")
    print("=" * 60)
    print("  Type your message and press Enter. Type 'done' to exit.")
    print("-" * 60)

    # ── Greeting turn ─────────────────────────────────────────────────────────
    greeting = invoke_graph(
        thread_id,
        "Greet the customer warmly and ask how you can help.",
    )
    logger.log("assistant", greeting)
    print(f"\n  Alex: {greeting}\n")

    # ── Conversation loop ──────────────────────────────────────────────────────
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("done", "quit", "exit", "bye", "goodbye"):
            break

        intent = detect_intent(user_input)
        logger.log("user", user_input, intent=intent)

        # ── Augment message with order / product / support context ─────────
        if intent == "order_status":
            note      = lookup_order(user_input)
            augmented = (
                f"{user_input}\n\n{note}" if note else
                f"{user_input}\n\n[SYSTEM NOTE: No order number found. "
                "Politely ask the customer for their order number (format: ORD-XXXX).]"
            )

        elif intent == "product_info":
            augmented = f"{user_input}\n\n{build_product_context()}"

        elif intent in ("returns", "billing", "account"):
            augmented = (
                f"{user_input}\n\n[SYSTEM NOTE: Intent = {intent}. "
                "Escalate if account access is required; "
                "mention 1-business-day email follow-up.]"
            )

        else:
            augmented = user_input

        # ── Send to graph — MemorySaver restores full history via thread_id ─
        reply = invoke_graph(thread_id, augmented)
        logger.log("assistant", reply)

        print(f"\n  Alex: {reply}\n")
        print("-" * 60)

    # ── Closing turn ──────────────────────────────────────────────────────────
    closing = invoke_graph(
        thread_id,
        "The customer is leaving. Give a warm 1-sentence goodbye.",
    )
    logger.log("assistant", closing)
    print(f"\n  Alex: {closing}\n")
    print("=" * 60)

    logger.save()

    # ── Show what MemorySaver stored for this thread ───────────────────────
    snapshot  = GRAPH.get_state({"configurable": {"thread_id": thread_id}})
    msg_count = len(snapshot.values["messages"])
    print(f"\n  [MemorySaver] {msg_count} messages checkpointed for thread '{thread_id}'")


# ── Entry Point ───────────────────────────────────────────────────────────────
while True:
    run_chat_session()
    again = input("\n  Start a new support session? (yes / no): ").strip().lower()
    if again not in ("yes", "y"):
        print(f"\n  Thank you for contacting {COMPANY_CONFIG['name']} support. Have a great day!\n")
        break

  Acme Corp  -  Customer Support & Sales
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
  Session ID (MemorySaver thread): 20260323_004025
  Type your message and press Enter. Type 'done' to exit.
------------------------------------------------------------

  Alex: Hello there! I'm Alex from Acme Corp, and I'd be delighted to assist you today. How can I help you get started?

You: I am currently looking for an order that is being shipped

  Alex: I can certainly help you with that! To check the status of your shipment, could you please provide me with your order number? It should start with "ORD-" followed by four digits, for example, ORD-1234.

------------------------------------------------------------
You: ORD-1001

  Alex: Thank you for providing your order number, ORD-1001! I've just checked, and it looks like your AlphaBook Pro Laptop has already shipped. The estimated delivery date is March 21, 2026, and it's being sent via FedEx with tracking number FX9284710234.

Is